# Hafta 1 — Model Context Protocol: Kurulum ve Mimari

> [`src/mcp_core/`](../src/mcp_core/) **için kaynak not defteri.**
> Aşağıda başvurulan tüm uygulamalar depoya eklenmiştir.

## Öğrenme hedefleri

Bu not defterinin sonunda şunları yapabileceksiniz:

1. MCP'nin üzerinde çalıştığı JSON-RPC 2.0 zarfını açıklamak ve neden seçildiğini izah etmek.
2. MCP yaşam döngüsünü (`initialize` → `tools/list` → `tools/call` → `shutdown`) elle uygulamak.
3. JSON-Schema ile doğrulanan bir girdi sözleşmesine sahip bir araç tanımlamak.
4. Korumalı bir yerel dosya sistemini ve bir SQLite veritabanını salt okunur MCP kaynakları olarak sunmak.
5. Kendi sunucunuzu stdio üzerinden asgari bir istemciyle yönetmek.

## 1. Protokolün biçimsel tanımı

Model Context Protocol, **JSON-RPC 2.0** üzerinde ince bir katmandır. Her mesaj üç biçimden birindedir:

**İstek (Request)**:
```json
{ "jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {} }
```

**Yanıt** (başarılı):
```json
{ "jsonrpc": "2.0", "id": 1, "result": { "tools": [ ... ] } }
```

**Yanıt** (hata):
```json
{ "jsonrpc": "2.0", "id": 1, "error": { "code": -32601, "message": "Method not found" } }
```

**Bildirim (Notification)** — `id` içermeyen ve yanıt beklemeyen bir istek.

### Neden REST veya gRPC değil de JSON-RPC?

| Özellik | JSON-RPC 2.0 | REST | gRPC |
|----------|--------------|------|------|
| Tek kanal üzerinden çift yönlü | ✓ | ✗ | ✓ |
| Aktarımdan bağımsız (stdio, ws, http) | ✓ | kısmen | ✗ (HTTP/2) |
| Şemasız kablo biçimi | ✓ | ✓ | ✗ (Protobuf) |
| Akış (streaming) yanıtları | ✓ bildirimlerle | SSE gerekir | ✓ |

MCP'nin hem stdio üzerinde (masaüstü bir LLM istemcisi sunucuyu alt süreç olarak başlatabilsin diye) **hem de** websocket üzerinde (barındırılan bir model ağ üzerinden bağlanabilsin diye) çalışması gerekir. Bu nedenle belirleyici etken JSON-RPC'nin aktarımdan bağımsız olmasıdır.

## 2. Yaşam döngüsü ve yetenek müzakeresi

Uyumlu bir alışveriş şöyle görünür:

```
İstemci                               Sunucu
  │                                     │
  │── initialize(protocolVersion) ──────▶
  │◀── result(serverInfo, capabilities)─│
  │                                     │
  │── notifications/initialized ────────▶  (yanıt yok)
  │                                     │
  │── tools/list ───────────────────────▶
  │◀── result({tools: [...]}) ──────────│
  │                                     │
  │── tools/call({name, arguments}) ────▶
  │◀── result({content: [...]}) ────────│
```

`initialize` alışverişi, iki tarafın bir protokol sürümünde anlaştığı ve sunucunun istemciye hangi isteğe bağlı yetenekleri (`tools`, `resources`, `prompts`, `sampling`, …) uyguladığını bildirdiği yerdir. Bu not defteri `tools` ve `resources` yeteneklerini uygular.

## 3. Kodda JSON-RPC zarfı

Uygulamanın tamamı [`src/mcp_core/protocol.py`](../src/mcp_core/protocol.py) dosyasındadır. İki temel öğe `JsonRpcRequest` ve `JsonRpcResponse` Pydantic modelleridir.

In [ ]:
from src.mcp_core.protocol import (
    JsonRpcRequest, JsonRpcResponse, JsonRpcError, MCPError,
    INVALID_PARAMS, METHOD_NOT_FOUND,
)

# Doğrudan bir istek oluştur.
req = JsonRpcRequest(id=1, method="tools/list", params={})
print(req.model_dump_json(exclude_none=True))

# Bildirimler id taşımaz.
notif = JsonRpcRequest(method="notifications/initialized")
print("is_notification:", notif.is_notification)


**Model tarafından zorlanan şema değişmezleri:**

- Bir yanıtta `result` ve `error` birbirini dışlar. Pydantic doğrulayıcısı ikisini birden ayarlayan ya da hiçbirini ayarlamayan her nesneyi reddeder.
- `id` alanı `int | str | None` tipindedir. `None` değerli bir id, mesajı bildirim olarak sınıflandırır.
- `-32099` ile `-32000` arasındaki hata kodları *uygulama tanımlı* hatalara ayrılmıştır; MCP uygulamaya özgü kodları (ör. `TOOL_NOT_FOUND = -32001`) bu aralığa yerleştirir.

## 4. Bir araç tanımlamak: şema sözleşmedir

Bir araç üç şeyden oluşur: bir ad, girdilerini tanımlayan bir JSON-Schema ve bir Python çağrılabilir nesnesi. Şema, bir LLM'in aracı çağırıp *çağırmayacağına* karar vermek için okuduğu şeydir; çağrılabilir nesne ise çağırdığında çalışan şeydir.

In [ ]:
from src.mcp_core.tool_registry import Tool, ToolRegistry

# Oyuncak bir sıcaklık dönüştürme aracı.
def celsius_to_fahrenheit(celsius: float) -> str:
    return f"{(celsius * 9 / 5) + 32:.2f} F"

c2f = Tool(
    name="celsius_to_fahrenheit",
    description="Bir sıcaklığı Celsius'tan Fahrenheit'a dönüştürür.",
    input_schema={
        "type": "object",
        "properties": {
            "celsius": {"type": "number", "description": "Celsius cinsinden sıcaklık."}
        },
        "required": ["celsius"],
        "additionalProperties": False,
    },
    handler=celsius_to_fahrenheit,
)

print(c2f.invoke({"celsius": 100}))      # -> "212.00 F"


In [ ]:
# Şema doğrulaması işleyici çalışmadan *önce* devreye girer.
try:
    c2f.invoke({"celsius": "sayı değil"})
except MCPError as exc:
    print("reddedildi:", exc.code, exc.message)


Bunun önemi şudur: bir LLM bir `Action Input` JSON nesnesi ürettiğinde, JSON sözleşmeye uymuyorsa işleyicinin bir `TypeError` ile çökmesini değil, yüksek sesle ve toparlanabilir biçimde başarısız olmasını isteriz. Şema doğrulayıcısı bize, ajanın kendini düzeltmesi için modele geri besleyebileceği yapılandırılmış bir `INVALID_PARAMS` hatası verir.

## 5. Sunucuyu süreç içinde ayağa kaldırmak

stdio döngüsünü not defterinde çalıştırmayacağız (çünkü stdin üzerinde bloklanır), ancak `handle()` yöntemine JSON çerçeveleri vererek sunucuyu doğrudan yönetebiliriz. Test paketinin yaptığı da tam olarak budur.

In [ ]:
import json
from src.mcp_core.server import MCPServer

registry = ToolRegistry()
registry.register(c2f)
server = MCPServer(name="weather-server", version="0.1.0", tools=registry)

def call(method, params=None, _id=[1]):
    req = {"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}}
    _id[0] += 1
    raw = server.handle(json.dumps(req))
    return json.loads(raw) if raw else None

print(call("initialize"))
print(call("tools/list"))
print(call("tools/call", {"name": "celsius_to_fahrenheit", "arguments": {"celsius": 0}}))


**Üç şeye dikkat edin:**

1. `initialize` yanıtı, sunucunun hangi yetenekleri desteklediğini duyurur — bu bilgi kayıtlı araçlardan ve kaynaklardan dinamik olarak doldurulur.
2. `tools/list`, tanımlayıcıyı *JSON-Schema ile birlikte* döndürür. Bir LLM bunu okuyup Python kaynağını görmeden aracı nasıl çağıracağı hakkında akıl yürütebilir.
3. `tools/call`, işleyicinin dönüş değerini tipli blokların yer aldığı bir `content` dizisinin içine sarar. Bu, MCP sunucularının metin, görüntü veya ikili verileri tek tip biçimde döndürebilmesini yansıtır.

## 6. Kaynaklar: URI ile adreslenebilen salt okunur veriler

Bir *kaynak* (resource), istemcinin bağlama okuyabileceği bir öğedir. Protokol 100 sayfalık bir PDF ile 10 satırlık bir veritabanı tablosu arasında ayrım yapmaz; her ikisi de bir MIME türüne sahip, URI ile adreslenebilen bayt akışlarıdır.

İki kaynak sağlayıcısı uyguluyoruz:

- **`FilesystemResources`** — bir korumalı alan kökünün altındaki dosyalar; sembolik bağ kaçışlarını reddeden bir `commonpath` denetimiyle.
- **`SQLiteResources`** — salt okunur bir SQLite veritabanının tabloları; her biri `sqlite://<tablo>` olarak sunulur.

In [ ]:
import tempfile, os, sqlite3, shutil
from src.mcp_core.resources import FilesystemResources, SQLiteResources

# 1. Dosya sistemi sağlayıcısı: iki dosya içeren geçici bir dizin oluştur.
tmp_root = tempfile.mkdtemp()
with open(f"{tmp_root}/notes.md", "w") as f:
    f.write("# Proje notları\n\nRAG, parçalar sorgu uzunluğuyla eşleştiğinde en iyi sonucu verir.")
os.makedirs(f"{tmp_root}/subdir", exist_ok=True)
with open(f"{tmp_root}/subdir/todo.txt", "w") as f:
    f.write("- MCP not defterini yaz\n- testleri ekle\n")

fs = FilesystemResources(tmp_root)
for r in fs.list_resources():
    print(r)

# Birini geri oku.
print("---")
print(fs.read("file://notes.md"))


In [ ]:
# 2. Korumalı alandan kaçış denetimi: ../ yolları ve sembolik bağlar reddedilir.
from src.mcp_core.protocol import MCPError
try:
    fs.read("file://../../etc/passwd")
except MCPError as exc:
    print("engellendi:", exc.message)

shutil.rmtree(tmp_root)


In [ ]:
# 3. SQLite sağlayıcısı: küçük bir veritabanı oluştur ve MCP üzerinden bir tablo oku.
db_path = tempfile.mktemp(suffix=".sqlite")
conn = sqlite3.connect(db_path)
conn.execute("CREATE TABLE papers (id TEXT, title TEXT)")
conn.executemany(
    "INSERT INTO papers VALUES (?, ?)",
    [("2212.10496", "Precise Zero-Shot Dense Retrieval (HyDE)"),
     ("1810.04805", "BERT: Pre-training of Deep Bidirectional Transformers")],
)
conn.commit(); conn.close()

db = SQLiteResources(db_path)
print(db.list_resources())
print(db.read("sqlite://papers"))


**Bunun ajanlar için önemi.** Hafta 6 bitirme projesi, aldığı arXiv makalelerini sunmak için SQLite kaynak sağlayıcısını kullanır; böylece LLM üst veri tablosunu geçici bir araç üzerinden değil, *bir kaynak olarak* sorgulayabilir. "Şu olguları bul" için doğru soyutlama kaynaklardır; "şu eylemi yap" için doğru soyutlama ise araçlardır.

## 7. Sunucuyu bir istemciden yönetmek

Depo, asgari bir stdio istemcisi olan [`src/mcp_core/client.py`](../src/mcp_core/client.py) dosyasını içerir. Kullanmak için sunucuyu bir alt süreç olarak başlatırsınız:

```bash
python -m src.mcp_core.server --config configs/mcp_server.yaml
```

…ve ardından bağlanırsınız:

```python
from src.mcp_core.client import MCPStdioClient
client = MCPStdioClient(["python", "-m", "src.mcp_core.server",
                        "--config", "configs/mcp_server.yaml"])
client.initialize()
print(client.list_tools())
print(client.call_tool("celsius_to_fahrenheit", {"celsius": 25}))
client.close()
```

Claude Desktop'ın sunucunuzla konuştuğu kablo protokolü de tam olarak budur. MCP uyumlu herhangi bir istemci (resmî SDK'lar, IDE içi asistanlar, kendi Python betiğiniz) araçlarınızı çağırabilir ve kaynaklarınızı okuyabilir.

## 8. Alıştırmalar

1. **Yazılabilir bir kaynak ekleyin.** `FilesystemResources` sınıfını `resources/write` destekleyecek şekilde genişletin (sunucuda yeni bir yöntem işleyicisine ihtiyacınız olacak). Güvenlik açısından sonuçlarının neler olduğunu — ve MCP'nin neden yazma için okumadan çok daha yüksek bir çıta koyduğunu — tartışın.
2. **Üçüncü bir yetenek ekleyin.** `prompts/list` ve `prompts/get` yöntemlerini uygulayarak küçük bir parametreli istem şablonu kütüphanesi döndürün. (İpucu: bir *istem şablonu*, işleyicisi dize ikamesi yapan bir araçtan ibarettir.)
3. **Protokol sürümünü müzakere edin.** `initialize` yönteminin, `protocolVersion` alanı sunucununkiyle uyumsuz olan bir istemciyi reddetmesini sağlayın. Doğru *kafes yapısı* hangisidir — anlamsal sürüm (semver) uyumluluğu mu, birebir eşleşme mi?

## 9. Çıkarımlar

- MCP, **sabit bir yöntem kümesine sahip JSON-RPC 2.0**'dır. JSON-RPC'yi ayrıştırıp dağıtabildiğinizde MCP küçük bir uzantıdır.
- **Şema sözleşmedir.** İşleyicileri çağırmadan önce girdileri doğrulayın, hataları yapılandırılmış biçimde iletin; ajanlarınız da zarifçe toparlansın.
- **Araçlar eylem içindir, kaynaklar olgular içindir.** Salt okunur veriler bir araç sarmalayıcısının içinde değil, `resources/read` arkasında yaşamalıdır.
- Aktarım (bizim durumumuzda stdio) **ikincildir**; aynı sunucu, G/Ç döngüsü değiştirilerek WebSocket veya HTTP üzerinden sunulabilir.

➡ Gelecek hafta bu temelleri, MCP'ye bağlı bir LLM'in sorgulayabileceği hiyerarşik bir erişim hattı kurmak için kullanacağız.